# Left-Wing Non-Fiction Books — Open Library API
Fetches books by selected left-wing authors and saves results to `Data/Raw/non fiction/leftpolitics_raw.csv`.

In [1]:
import requests
import pandas as pd
import time

In [2]:
AUTHORS = [
    # Already included
    "Angela Davis",
    "James Baldwin",
    "Mahmoud Darwish",
    "Noam Chomsky",
    "Naomi Klein",
    "bell hooks",
    "Howard Zinn",
    "Eduardo Galeano",
    "Frantz Fanon",
    "Rosa Luxemburg",
    "Antonio Gramsci",
    "Michel Foucault",
    "Cornel West",
    "Ta-Nehisi Coates",
    "Robin D.G. Kelley",

    # Black American & diaspora thinkers
    "W.E.B. Du Bois",
    "Audre Lorde",
    "Malcolm X",
    "Cedric Robinson",
    "Saidiya Hartman",
    "Keeanga-Yamahtta Taylor",
    "Patricia Hill Collins",
    "Manning Marable",
    "Grace Lee Boggs",
    "Ibram X. Kendi",

    # African & Caribbean theorists
    "C.L.R. James",
    "Aimé Césaire",
    "Walter Rodney",
    "Achille Mbembe",
    "Stuart Hall",

    # Latin American & Global South
    "Paulo Freire",
    "Vijay Prashad",
    "José Carlos Mariátegui",

    # Feminist & queer theory
    "Sara Ahmed",
    "Silvia Federici",
    "Gayatri Chakravorty Spivak",

    # Contemporary left
    "David Graeber",
    "Mark Fisher",
    "Slavoj Žižek",
]

In [3]:
SEARCH_URL = "https://openlibrary.org/search.json"


def fetch_books_for_author(author_name, max_results=50):
    params = {
        "author": author_name,
        "fields": "title,author_name,first_publish_year,subject,edition_count,key",
        "limit": max_results,
    }
    try:
        resp = requests.get(SEARCH_URL, params=params, timeout=15)
        resp.raise_for_status()
        docs = resp.json().get("docs", [])
    except requests.RequestException as e:
        print(f"  Error fetching '{author_name}': {e}")
        return []

    records = []
    for doc in docs:
        authors = doc.get("author_name", [])
        if not any(author_name.lower() in a.lower() for a in authors):
            continue
        subjects = doc.get("subject", [])
        records.append({
            "title": doc.get("title", ""),
            "author": ", ".join(authors),
            "year_published": doc.get("first_publish_year"),
            "subjects": " | ".join(subjects[:10]) if subjects else "",
            "edition_count": doc.get("edition_count", 0),
            "open_library_key": doc.get("key", ""),
            "queried_author": author_name,
        })
    return records

In [4]:
all_records = []

for author in AUTHORS:
    print(f"Fetching: {author}...")
    books = fetch_books_for_author(author)
    print(f"  → {len(books)} books found")
    all_records.extend(books)
    time.sleep(0.5)

print(f"\nTotal records collected: {len(all_records)}")

Fetching: Angela Davis...
  → 4 books found
Fetching: James Baldwin...
  → 33 books found
Fetching: Mahmoud Darwish...
  → 50 books found
Fetching: Noam Chomsky...
  → 50 books found
Fetching: Naomi Klein...
  → 19 books found
Fetching: bell hooks...
  → 50 books found
Fetching: Howard Zinn...
  → 50 books found
Fetching: Eduardo Galeano...
  → 50 books found
Fetching: Frantz Fanon...
  → 24 books found
Fetching: Rosa Luxemburg...
  → 50 books found
Fetching: Antonio Gramsci...
  → 50 books found
Fetching: Michel Foucault...
  → 50 books found
Fetching: Cornel West...
  → 50 books found
Fetching: Ta-Nehisi Coates...
  → 50 books found
Fetching: Robin D.G. Kelley...
  → 44 books found
Fetching: W.E.B. Du Bois...
  → 0 books found
Fetching: Audre Lorde...
  → 49 books found
Fetching: Malcolm X...
  → 50 books found
Fetching: Cedric Robinson...
  → 11 books found
Fetching: Saidiya Hartman...
  → 0 books found
Fetching: Keeanga-Yamahtta Taylor...
  → 10 books found
Fetching: Patricia Hill 

In [5]:
df = pd.DataFrame(all_records)

df = df.drop_duplicates(subset=["title", "author"])
df = df.sort_values(["queried_author", "year_published"], na_position="last")
df = df.reset_index(drop=True)

print(f"Unique records after deduplication: {len(df)}")
df.head(10)

Unique records after deduplication: 1316


,title,author,year_published,subjects,edition_count,open_library_key,queried_author
0,Le PROBLÈME NATIONAL KAMERUNAIS,Achille Mbembe,1984.0,Politics and government | Nationalism | Nation...,1,/works/OL31757685W,Achille Mbembe
1,Les jeunes et l'ordre politique en Afrique noire,Achille Mbembe,1985.0,Youth | Political activity,1,/works/OL2537390W,Achille Mbembe
2,Les JEUNES ET L'ORDRE POLITIQUE EN AFRIQUE NOIRE,Achille Mbembe,1985.0,Youth | Political activity,1,/works/OL31741886W,Achille Mbembe
3,Afriques indociles,Achille Mbembe,1988.0,Afrique | Religion et État | Christianisme | C...,1,/works/OL9171632W,Achille Mbembe
4,"La naissance du maquis dans le Sud-Cameroun, 1...",Achille Mbembe,1996.0,History | Insurgency | Politics and government...,2,/works/OL2537389W,Achille Mbembe
5,Du gouvernement privé indirect,Achille Mbembe,1999.0,Politics and government | Privatization,1,/works/OL2537388W,Achille Mbembe
6,De la postcolonie,Achille Mbembe,2000.0,Politics and government | Social conditions | ...,9,/works/OL2537387W,Achille Mbembe
7,On private indirect government,Achille Mbembe,2000.0,Politics and government | Privatization,1,/works/OL2537391W,Achille Mbembe
8,Globalization,"Arjun Appadurai, Achille Mbembe, Philippe Reka...",2001.0,Globalization | International relations,2,/works/OL25363313W,Achille Mbembe
9,On Private Indirect Government,Achille Mbembe,2002.0,"Africa, sub-saharan, politics and government |...",1,/works/OL9171634W,Achille Mbembe


In [6]:
import os

output_dir = "Data/Raw/non fiction"
output_path = os.path.join(output_dir, "leftpolitics_raw.csv")

os.makedirs(output_dir, exist_ok=True)
df.to_csv(output_path, index=False)
print(f"Saved {len(df)} rows to {output_path}")

Saved 1316 rows to Data/Raw/non fiction/leftpolitics_raw.csv
